# 02 — Pandas cross-check for Task 2.2

Reproduces the SQL clinic counts and no-show counts with `groupby` on `appointments_clean.csv`, and asserts they match the values the SQL query returned.

In [ ]:
import pandas as pd
from pathlib import Path

root = Path.cwd().parent                 
appointments = pd.read_csv(root / "appointments_clean.csv")
clinics = pd.read_csv(root / "clinics_enriched.csv")

In [ ]:
by_clinic = (
    appointments.merge(clinics[["clinic_id", "clinic_name", "city"]], on="clinic_id")
    .groupby(["clinic_name", "city"])
    .agg(appointments=("no_show", "size"), no_shows=("no_show", "sum"))
    .assign(no_show_pct=lambda t: (100 * t["no_shows"] / t["appointments"]).round(1))
    .sort_values("no_show_pct", ascending=False)
    .reset_index()
)
by_clinic

Counts returned by the SQL query in Task 2.2.2 (all 8 clinics, no `HAVING`):

In [ ]:
sql_result = pd.DataFrame(
    [("Gomti Nagar Care Point", 90, 30), ("Hadapsar Care Centre", 143, 39),
     ("Malviya Nagar Clinic", 142, 38), ("Kothrud Family Clinic", 160, 36),
     ("Vijay Nagar Clinic", 101, 21), ("Edappally Medical Centre", 142, 23),
     ("Sitabuldi Health Point", 82, 13), ("Vaishali Nagar Clinic", 130, 18)],
    columns=["clinic_name", "appointments", "no_shows"],
)

check = by_clinic.merge(sql_result, on="clinic_name", suffixes=("_pandas", "_sql"))
matches = (check["appointments_pandas"] == check["appointments_sql"]) & (check["no_shows_pandas"] == check["no_shows_sql"])
print(f"Clinics matching SQL: {matches.sum()} / {len(check)}")
assert matches.all() and len(check) == 8, "pandas and SQL counts differ"
print("pandas counts match the SQL counts exactly.")

In [ ]:
print("At least 95 appointments:")
by_clinic[by_clinic["appointments"] >= 95]